# Sprint 1 Campus — From application to service

Work through this notebook alongside the Campus lessons. Each code cell runs the supplied FieldCare FastAPI application or a clearly labelled worked example. The request, route, validation, and service code stay visible; no helper prints source files or invents a result.

Save your own copy in Drive. At the end, download the service checkpoint ZIP. If Colab restarts, upload that ZIP and enter its path in setup.

## Campus lessons

1. [Application vs service](https://app.notion.com/p/3ea5a55972ad81a09e8feba7b4bb576a?pvs=204) and [REST in plain language](https://app.notion.com/p/3ea5a55972ad813b906bd2f6edb69740?pvs=204)
2. [Touring the provided FastAPI service](https://app.notion.com/p/3ea5a55972ad81cd844dccbe5d8b0b89?pvs=204)
3. [Extending the scaffold for a new use case](https://app.notion.com/p/3ea5a55972ad811e9461cc2e184edeb6?pvs=204)
4. [Why contracts matter](https://app.notion.com/p/3ea5a55972ad812c8fcaf9462d1a2b9e?pvs=204)
5. [Why and how to version endpoints](https://app.notion.com/p/3ea5a55972ad815795d0e07af30135e7?pvs=204)
6. [Service scaffold checkpoint](https://app.notion.com/p/3ea5a55972ad8180aabdcc14a959c3d2?pvs=204)

Supported requests use the Module A model path: current service documents are indexed in Chroma with OpenRouter embeddings, LangGraph retrieves evidence and coordinates generation, and OpenRouter returns the model answer. Set `OPENROUTER_API_KEY` in Colab Secrets (or use the hidden prompt) before the first request. Model output varies; validation and clarification branches are deterministic service behavior.

**Colab runtime:** These Sprint 1 exercises use FastAPI `TestClient` in-process, so they do not start Uvicorn in the notebook’s foreground cell. Do not add `uvicorn.run()` or an ngrok tunnel; `uvicorn.run()` keeps its cell occupied until the server stops. Later sprints use the Module B `ServiceProcess` helper to start one loopback-only Uvicorn child, check readiness with a timeout, and stop only that owned process. The request cells wait for their real HTTP/provider response; the server itself does not hold the cell open.


## Campus lessons in Notion

- [Application vs service](https://app.notion.com/p/3ea5a55972ad81a09e8feba7b4bb576a?pvs=204)
- [REST in plain language](https://app.notion.com/p/3ea5a55972ad813b906bd2f6edb69740?pvs=204)
- [Touring the provided FastAPI service](https://app.notion.com/p/3ea5a55972ad81cd844dccbe5d8b0b89?pvs=204)
- [Extending the scaffold for a new use case](https://app.notion.com/p/3ea5a55972ad811e9461cc2e184edeb6?pvs=204)
- [Why contracts matter for AI services](https://app.notion.com/p/3ea5a55972ad812c8fcaf9462d1a2b9e?pvs=204)
- [Why and how to version AI endpoints](https://app.notion.com/p/3ea5a55972ad815795d0e07af30135e7?pvs=204)
- [Service scaffold checkpoint](https://app.notion.com/p/3ea5a55972ad8180aabdcc14a959c3d2?pvs=204)


In [ ]:
# Setup: run once in a fresh Colab runtime.
from pathlib import Path
from shutil import copytree
from zipfile import ZipFile
import os
import subprocess
import sys

REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
REPO = Path("/content/ai-app-dev_Mod-B")
PROJECT = Path("/content/fieldcare")
CHECKPOINT = ""  # Optional uploaded ZIP, e.g. /content/sprint-1-campus-checkpoint.zip

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
sys.path.insert(0, str(REPO / "src"))  # Expose the installed source to this running kernel.

from module_b.openrouter import DEFAULT_CHAT_MODEL, require_openrouter_key
require_openrouter_key(prompt=True)
os.environ.setdefault("OPENROUTER_MODEL", DEFAULT_CHAT_MODEL)

if CHECKPOINT:
    PROJECT.mkdir(parents=True, exist_ok=True)
    with ZipFile(CHECKPOINT) as archive:
        archive.extractall(PROJECT)
elif not PROJECT.exists():
    copytree(REPO / "examples" / "fieldcare", PROJECT)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
client.__enter__()  # Start the app lifespan and initialize its real Chroma/OpenRouter resources.
print("Workspace:", PROJECT)

## 1. Application, service, and REST

An application is used directly by a person. A service exposes an operation that another program can call. Here the notebook is the caller and FieldCare is the service.

An endpoint combines an HTTP method and a path. The request body carries the caller’s data; the response has a status code and a body. Predict which request needs a JSON body, then run both.

In [ ]:
health = client.get("/health")
question = {
    "question": "Which filter and airflow checks are documented?",
    "equipment_id": "EQ-FC-1002",
}
diagnosis = client.post("/v1/diagnose", json=question)

print("GET /health:", health.status_code, health.json())
print("POST /v1/diagnose:", diagnosis.status_code, diagnosis.json())

The request went through the supplied FastAPI route and FieldCare code. The returned `mode` identifies the configured live path. A successful status tells you the service handled the request; review the generated answer and its cited evidence before relying on it.

Open app/main.py, app/routes.py, app/schemas.py, app/service.py, and app/model.py in Colab’s Files panel. Follow the route to the input and output declarations, then to the application and model adapter. Read the source directly; this notebook does not copy it into cell output.

## 2. Tour the request path

The request model requires a non-empty question and allows equipment_id to be omitted. Run these requests and compare invalid input with valid input that needs clarification.

In [ ]:
cases = [
    ("supported", question),
    ("invalid: required question missing", {}),
    ("valid: optional equipment context missing",
     {"question": "Which airflow checks are documented?"}),
]
for label, payload in cases:
    result = client.post("/v1/diagnose", json=payload)
    print(label, "→", result.status_code, result.json())

FastAPI rejects a malformed request before the route runs. A valid request without equipment context can reach FieldCare’s clarification path. For a supported request, trace the actual code:

1. app/routes.py receives the request.
2. app/schemas.py declares the request and response fields.
3. app/service.py checks scope and gathers evidence.
4. app/orchestration.py runs retrieval and generation in LangGraph; app/model.py sends the generation request through the Module A OpenRouter client.
5. The route returns a DiagnosticResponse.

A request for smoke, fire, sparks, or another unsupported condition follows a safety clarification path. The starter is limited to its published filter and airflow evidence.

## 3. Extend the scaffold

In your project files, add a route with its own prompt for dispatch. Reuse DiagnosticRequest and DiagnosticResponse. Build a route-specific LangGraph with the shared Chroma store and OpenRouter client so this route uses its own prompt and approved model. Register the route in app/main.py. The Campus lesson defines the handover requirements and provides the pattern to adapt.

Save the new file as `app/handover_routes.py` inside the FieldCare workspace shown by setup. Do not replace the technician route or add a hardcoded answer. The next cell checks that the file exists, refreshes Python’s import cache, and reloads your route and app before sending actual requests to both routes.

In [ ]:
# Confirm the new file is in the FieldCare workspace, then reload the route and app.
from importlib import invalidate_caches, reload

HANDOVER_FILE = PROJECT / "app" / "handover_routes.py"
if not HANDOVER_FILE.is_file():
    raise FileNotFoundError(
        f"Create and save your route at {HANDOVER_FILE} before running this cell."
    )

# Colab may not notice a file added after Python first scanned the app directory.
invalidate_caches()
import app.handover_routes as handover_routes
import app.main as main

reload(handover_routes)
reload(main)
if client is not None:
    client.__exit__(None, None, None)
client = TestClient(main.app)
client.__enter__()  # Initialize the app resources for this reloaded application.

HANDOVER_PATH = "/v1/dispatch-handover"
handover_request = {
    "question": "The unit runs hot after filter service. Which checks should be verified?",
    "equipment_id": "EQ-FC-1002",
}

for path in ["/v1/diagnose", HANDOVER_PATH]:
    result = client.post(path, json=handover_request)
    print(path, "→", result.status_code, result.json())


### Compare the actual route outputs

The previous request used the shared OpenRouter model through each route's own configuration. Compare the real responses and citations. Wording can vary between calls, so use the prompt and retrieved evidence—not one lucky phrase—to explain the difference.


## 4. Contracts for callers

A contract describes accepted input and the response a caller can rely on. A schema makes structural requirements executable. The code below applies the supplied Pydantic model to actual values.

In [ ]:
from pydantic import ValidationError
from app.schemas import DiagnosticRequest, DiagnosticResponse

for payload in [
    question,
    {"question": "   ", "equipment_id": "EQ-FC-1002"},
    {**question, "unexpected": True},
]:
    try:
        parsed = DiagnosticRequest.model_validate(payload)
        print("Accepted:", parsed.model_dump())
    except ValidationError as error:
        print("Rejected:", error.errors()[0]["loc"], error.errors()[0]["type"])

actual_response = DiagnosticResponse.model_validate(diagnosis.json())
print("Actual response satisfies the declared output schema:", actual_response.model_dump())

malformed_response = {**diagnosis.json(), "citations": "not-a-list"}
try:
    DiagnosticResponse.model_validate(malformed_response)
except ValidationError as error:
    print("Malformed output rejected:", error.errors()[0]["loc"], error.errors()[0]["type"])


The real response satisfies the declared output shape. The deliberately malformed `citations` value fails structural validation. A structurally valid response can still make an unsupported claim, so grounding requires reviewing the model output against its retrieved evidence. The malformed example is a schema check; it is not a model response.

A `422` input rejection, a valid clarification, and a model-backed response describe different outcomes. Keep those distinctions visible when documenting your own route.


## 5. Version endpoints

A versioned path lets callers choose an agreement. In this worked example, the original diagnostic request accepts up to 2,000 characters and a pilot accepts up to 600. Both routes use the same FieldCare application, but the pilot has its own request model, prompt, and model setting.

Run the next cell to define a real FastAPI route in this notebook runtime.

In [ ]:
from os import getenv

from fastapi import APIRouter, Request
from pydantic import BaseModel, ConfigDict, Field
from app.orchestration import build_diagnosis_graph
from app.service import run_diagnosis
from app.schemas import DiagnosticResponse

class DiagnosticV2Request(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)
    question: str = Field(min_length=1, max_length=600)
    equipment_id: str | None = Field(default=None, min_length=1, max_length=64)

v2_router = APIRouter()
V2_PROMPT = (
    "Answer the supplied filter or airflow question using only the retrieved evidence. "
    "State what remains to verify; do not claim an inspection or repair occurred."
)
V2_MODEL = getenv("FIELDCARE_DIAGNOSE_V2_MODEL", DEFAULT_CHAT_MODEL).strip()

@v2_router.post("/v2/diagnose", response_model=DiagnosticResponse)
def diagnose_v2(payload: DiagnosticV2Request, request: Request):
    graph = build_diagnosis_graph(
        store=request.app.state.document_store,
        client=request.app.state.openrouter_client,
        system_prompt=V2_PROMPT,
        model_name=V2_MODEL,
    )
    return run_diagnosis(payload, graph=graph)

from app.main import app
app.include_router(v2_router)

short_request = question
long_request = {"question": "filter " + "x" * 700, "equipment_id": "EQ-FC-1002"}
for path in ["/v1/diagnose", "/v2/diagnose"]:
    for label, payload in [("short", short_request), ("long", long_request)]:
        result = client.post(path, json=payload)
        print(path, label, "→", result.status_code)


A deprecation marker tells callers to plan a move; it does not remove the route. Temporarily mark the original route, inspect FastAPI's OpenAPI description, and make a real request. The code restores the original metadata afterward.


In [ ]:
from fastapi.routing import APIRoute

diagnostic_route = next(
    route for route in app.routes
    if isinstance(route, APIRoute) and route.path == "/v1/diagnose"
)
was_deprecated = diagnostic_route.deprecated
try:
    diagnostic_route.deprecated = True
    app.openapi_schema = None
    print("OpenAPI marks /v1/diagnose deprecated:",
          app.openapi()["paths"]["/v1/diagnose"]["post"].get("deprecated", False))
    result = client.post("/v1/diagnose", json=question)
    print("Existing route still responds:", result.status_code, result.json())
finally:
    diagnostic_route.deprecated = was_deprecated
    app.openapi_schema = None


The original route accepts the longer request; the pilot rejects it. That difference comes from their separate schemas. Each route also binds its own prompt and approved model setting to the same Chroma evidence and LangGraph workflow. The status and route configuration demonstrate compatibility; generated wording varies and is not the versioning proof.

The deprecation marker appears in OpenAPI while the old route still responds. In production, callers need a migration period before removal.


## 6. Service scaffold checkpoint
Extend your service for a shift supervisor who needs a briefing about one equipment question. In your project files, create a new endpoint, define its input/output contract, and give it a versioned path. Choose the route name, question limit, prompt, and model-setting name. The response must follow `DiagnosticResponse` (`answer`, `status`, `citations`, and `mode`).

Use the supplied FieldCare application, shared Chroma evidence store, and LangGraph/OpenRouter workflow. Keep the earlier diagnostic and dispatch routes working. Do not claim a measurement, inspection, cause, or repair unless the retrieved evidence contains it. The generated wording must be reviewed against its cited sources; it is not fixed.

After adding and registering your route, reload the application and enter its TestClient lifespan again if Python has cached an earlier app version. Then run the request checks below using your chosen path and one malformed request.


In [ ]:
# Set your chosen supervisor endpoint and test its contract.
SUPERVISOR_PATH = "/v1/supervisor-briefing"
supervisor_request = {
    "question": "Which filter and airflow checks should the next technician verify?",
    "equipment_id": "EQ-FC-1002",
}
malformed_request = {"question": "", "equipment_id": "EQ-FC-1002"}

for path, payload in [
    ("/v1/diagnose", question),
    ("/v1/dispatch-handover", handover_request),
    (SUPERVISOR_PATH, supervisor_request),
    (SUPERVISOR_PATH, malformed_request),
]:
    result = client.post(path, json=payload)
    print(path, "→", result.status_code, result.json())
    if path == SUPERVISOR_PATH and result.status_code == 200:
        declared_response = DiagnosticResponse.model_validate(result.json())
        print("Declared output contract:", declared_response.model_dump())


Before you finish, explain what each observation establishes: that the route is callable, invalid input is rejected, the successful response passes the declared output model, and an earlier caller still works. A successful status does not establish model answer quality; compare the real generated answer with its retrieved source evidence.


## Save your Campus checkpoint

The ZIP contains the service’s app and data files. Save the notebook separately in Drive; it contains your notes and the request examples.

In [ ]:
from zipfile import ZIP_DEFLATED

CHECKPOINT_OUT = Path("/content/sprint-1-campus-checkpoint.zip")
with ZipFile(CHECKPOINT_OUT, "w", ZIP_DEFLATED) as archive:
    for folder in ["app", "data"]:
        for file in (PROJECT / folder).rglob("*"):
            if file.is_file() and "__pycache__" not in file.parts and file.suffix != ".pyc":
                archive.write(file, file.relative_to(PROJECT))
print("Download:", CHECKPOINT_OUT)

if client is not None:
    client.__exit__(None, None, None)
    client = None